# Data Cleaning and Preparation

In [1]:
import pandas as pd
import numpy as np
import os

## 1. Load Raw Data

In [2]:
customers = pd.read_csv("../data/raw/customers_raw.csv")
campaigns = pd.read_csv("../data/raw/campaigns_raw.csv")
responses = pd.read_csv("../data/raw/responses_raw.csv")
support_tickets = pd.read_csv("../data/raw/support_tickets_raw.csv")
transactions = pd.read_csv("../data/raw/transactions_raw.csv")

raw_row_counts = {
    'customers': len(customers),
    'campaigns': len(campaigns),
    'responses': len(responses),
    'support_tickets': len(support_tickets),
    'transactions': len(transactions),
}

## 2. Clean Customers Data
- Remove duplicate rows
- Standardize text columns (city, customer_segment)
- Handle invalid and missing emails
- Convert dates

In [3]:
# Remove exact duplicates, then keep one canonical record per customer ID
customers = customers.drop_duplicates().drop_duplicates(subset=['customer_id'], keep='first').copy()

# Standardize text
customers['name'] = customers['name'].str.strip()
customers['city'] = customers['city'].str.strip().str.title()
customers['customer_segment'] = customers['customer_segment'].str.strip().str.title()

# Fix invalid emails (set to NaN for now)
email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
invalid_email_mask = customers["email"].notna() & ~customers["email"].astype(str).str.match(email_pattern)
customers.loc[invalid_email_mask, 'email'] = np.nan

# Handle missing values (Fill or Drop based on business rules)
customers['city'] = customers['city'].fillna('Unknown')

# Convert dates; retain unparseable values as missing rather than propagating strings
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

print(f"Cleaned Customers Rows: {len(customers)}")
customers.head()

Cleaned Customers Rows: 5000


,customer_id,name,email,phone,city,signup_date,customer_segment
0,C00001,Hossain Ahmed,hossain.ahmed1@example.com,1.595310e+09,Rajshahi,2025-08-17,Standard
1,C00002,Hossain Sarker,hossain.sarker2@example.com,1.393350e+09,Rangpur,2025-02-02,Standard
2,C00003,Mim Rahman,mim.rahman3@example.com,1.842621e+09,Dhaka,2026-07-29,Standard
3,C00004,Ayesha Chowdhury,ayesha.chowdhury4@example.com,1.536696e+09,Mymensingh,2026-08-27,Standard
4,C00005,Rahim Hossain,rahim.hossain5@example.com,1.753779e+09,Barishal,2025-10-12,Standard


## 3. Clean Campaigns Data
- Remove duplicate rows
- Handle missing and negative budgets
- Convert dates

In [4]:
# Remove duplicates and standardize categorical values
campaigns = campaigns.drop_duplicates()
for column in ['channel', 'objective', 'status', 'target_audience']:
    campaigns[column] = campaigns[column].str.strip().str.title()

# Handle negative and missing budgets by imputing with the median valid budget
median_budget = campaigns.loc[campaigns['budget'] > 0, 'budget'].median()
campaigns.loc[campaigns['budget'].isna() | campaigns['budget'].le(0), 'budget'] = median_budget

# Invalid campaign dates cannot support time-based analysis
campaigns['campaign_date'] = pd.to_datetime(campaigns['campaign_date'], errors='coerce')
campaigns = campaigns.dropna(subset=['campaign_date']).copy()

print(f"Cleaned Campaigns Rows: {len(campaigns)}")
campaigns.head()

Cleaned Campaigns Rows: 39


,campaign_id,campaign_name,channel,campaign_date,budget,objective,status,target_audience
0,CMP001,New Year Promotion 01,Instagram,2026-03-15,50000.0,Conversion,Scheduled,Standard
1,CMP002,Product Launch 02,Sms,2026-09-13,75000.0,Awareness,Scheduled,All Customers
2,CMP003,Festival Offer 03,Instagram,2026-10-08,150000.0,Retention,Active,Premium
3,CMP004,New Year Promotion 04,Email,2026-03-05,50000.0,Engagement,Completed,Premium
4,CMP005,Autumn Rewards 05,Google Ads,2026-03-07,150000.0,Retention,Scheduled,Vip


## 4. Clean Responses Data
- Remove duplicates
- Drop rows with missing customer_id or campaign_id
- Convert dates

In [5]:
# Remove duplicates and standardize response types
responses = responses.drop_duplicates()
responses['response_type'] = responses['response_type'].str.strip().str.title()

# Keep only records with usable keys, dates, and binary conversion flags
responses['response_date'] = pd.to_datetime(responses['response_date'], errors='coerce')
responses = responses.dropna(subset=['customer_id', 'campaign_id', 'response_date'])
responses = responses[responses['converted'].isin([0, 1])].copy()

print(f"Cleaned Responses Rows: {len(responses)}")
responses.head()

Cleaned Responses Rows: 14920


,response_id,customer_id,campaign_id,response_date,response_type,converted
0,R000001,C03347,CMP034,2026-03-01,Clicked,0
1,R000002,C01448,CMP025,2026-01-15,Viewed,0
2,R000003,C01841,CMP008,2026-02-21,Clicked,0
3,R000004,C01284,CMP033,2026-06-05,Responded,0
4,R000005,C03101,CMP038,2026-06-16,Clicked,0


## 5. Clean Support Tickets Data
- Remove duplicates
- Fill missing categories
- Handle missing resolution dates and satisfaction scores
- Convert dates

In [6]:
# Remove duplicates and standardize categorical values
support_tickets = support_tickets.drop_duplicates()
for column in ['category', 'priority', 'status']:
    support_tickets[column] = support_tickets[column].str.strip().str.title()
support_tickets['category'] = support_tickets['category'].fillna('Uncategorized')

# Created dates are required; unresolved tickets retain missing resolution dates.
support_tickets['created_date'] = pd.to_datetime(support_tickets['created_date'], errors='coerce')
support_tickets['resolution_date'] = pd.to_datetime(support_tickets['resolution_date'], errors='coerce')
support_tickets = support_tickets.dropna(subset=['created_date']).copy()

# A resolution before creation is invalid, so preserve the ticket but mark it unresolved.
invalid_resolution_order = support_tickets['resolution_date'].lt(support_tickets['created_date'])
support_tickets.loc[invalid_resolution_order, 'resolution_date'] = pd.NaT
support_tickets.loc[~support_tickets['satisfaction_score'].between(1, 5), 'satisfaction_score'] = np.nan

print(f"Cleaned Support Tickets Rows: {len(support_tickets)}")
support_tickets.head()

Cleaned Support Tickets Rows: 8000


,ticket_id,customer_id,created_date,category,priority,status,resolution_date,satisfaction_score
0,T000001,C02228,2026-08-02,Account,Low,In Progress,NaT,NaN
1,T000002,C00593,2026-09-05,Payment,Medium,Open,NaT,NaN
2,T000003,C02549,2026-01-19,Order,Low,In Progress,NaT,NaN
3,T000004,C01263,2026-04-19,Other,High,Resolved,2026-04-24,4.0
4,T000005,C00469,2026-01-17,Delivery,Medium,Closed,2026-01-20,2.0


## 6. Clean Transactions Data
- Remove duplicates
- Fill missing values
- Convert dates

In [7]:
# Remove duplicates and fill descriptive missing values
transactions = transactions.drop_duplicates()
transactions['product'] = transactions['product'].fillna('Unknown').str.strip()
transactions['payment_method'] = transactions['payment_method'].fillna('Unknown').str.strip().str.title()
transactions['status'] = transactions['status'].str.strip().str.title()

# Keep only transactions with a valid date and a positive amount
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], errors='coerce')
transactions = transactions.dropna(subset=['transaction_date']).copy()
transactions = transactions[transactions['amount'].gt(0)].copy()

print(f"Cleaned Transactions Rows: {len(transactions)}")
transactions.head()

Cleaned Transactions Rows: 11935


,transaction_id,customer_id,transaction_date,product,amount,payment_method,status
0,TRX000001,C00626,2026-07-24,Premium Subscription,5531.72,Bkash,Completed
1,TRX000002,C01245,2026-10-26,Enterprise Plan,96039.96,Card,Completed
2,TRX000003,C02623,2026-01-28,Premium Subscription,2812.35,Nagad,Completed
3,TRX000004,C00762,2026-07-23,Data Insights Package,8405.68,Bank Transfer,Completed
4,TRX000005,C00609,2026-08-24,Enterprise Plan,71449.88,Bank Transfer,Completed


## 7. Relational Validation
Ensure all foreign keys exist in the primary tables.

In [8]:
valid_customers = set(customers['customer_id'])
valid_campaigns = set(campaigns['campaign_id'])

# Keep only records whose foreign keys exist.
responses = responses[
    responses['customer_id'].isin(valid_customers)
    & responses['campaign_id'].isin(valid_campaigns)
].copy()
support_tickets = support_tickets[support_tickets['customer_id'].isin(valid_customers)].copy()
transactions = transactions[transactions['customer_id'].isin(valid_customers)].copy()

# Enforce event chronology. Source rows remain unchanged in data/raw/.
customer_signup = customers.set_index('customer_id')['signup_date']
campaign_start = campaigns.set_index('campaign_id')['campaign_date']

response_temporal_mask = (
    responses['response_date'].ge(responses['customer_id'].map(customer_signup))
    & responses['response_date'].ge(responses['campaign_id'].map(campaign_start))
)
ticket_temporal_mask = support_tickets['created_date'].ge(
    support_tickets['customer_id'].map(customer_signup)
)
transaction_temporal_mask = transactions['transaction_date'].ge(
    transactions['customer_id'].map(customer_signup)
)

temporal_exclusion_report = pd.DataFrame([
    {'dataset': 'responses', 'rule': 'response on/after signup and campaign date', 'rows_excluded': int((~response_temporal_mask).sum())},
    {'dataset': 'support_tickets', 'rule': 'ticket on/after signup date', 'rows_excluded': int((~ticket_temporal_mask).sum())},
    {'dataset': 'transactions', 'rule': 'transaction on/after signup date', 'rows_excluded': int((~transaction_temporal_mask).sum())},
])

responses = responses[response_temporal_mask].copy()
support_tickets = support_tickets[ticket_temporal_mask].copy()
transactions = transactions[transaction_temporal_mask].copy()

print("Relational validation complete.")
print(f"Final Responses Rows: {len(responses)}")
print(f"Final Support Tickets Rows: {len(support_tickets)}")
print(f"Final Transactions Rows: {len(transactions)}")

Relational validation complete.
Final Responses Rows: 7477
Final Support Tickets Rows: 6601
Final Transactions Rows: 9862


## 8. Final Quality Check
Confirm that the exported data has no duplicate rows or broken foreign keys.

In [9]:
cleaned_datasets = {
    'customers': customers,
    'campaigns': campaigns,
    'responses': responses,
    'support_tickets': support_tickets,
    'transactions': transactions,
}

quality_report = pd.DataFrame({
    'dataset': cleaned_datasets.keys(),
    'raw_rows': [raw_row_counts[name] for name in cleaned_datasets],
    'clean_rows': [len(df) for df in cleaned_datasets.values()],
    'rows_removed': [raw_row_counts[name] - len(df) for name, df in cleaned_datasets.items()],
    'duplicate_rows': [df.duplicated().sum() for df in cleaned_datasets.values()],
    'missing_cells': [df.isna().sum().sum() for df in cleaned_datasets.values()],
})

assert customers['customer_id'].is_unique
assert campaigns['campaign_id'].is_unique
assert responses['response_id'].is_unique
assert support_tickets['ticket_id'].is_unique
assert transactions['transaction_id'].is_unique
assert responses['customer_id'].isin(customers['customer_id']).all()
assert responses['campaign_id'].isin(campaigns['campaign_id']).all()
assert support_tickets['customer_id'].isin(customers['customer_id']).all()
assert transactions['customer_id'].isin(customers['customer_id']).all()
assert responses['response_date'].ge(responses['customer_id'].map(customer_signup)).all()
assert responses['response_date'].ge(responses['campaign_id'].map(campaign_start)).all()
assert support_tickets['created_date'].ge(support_tickets['customer_id'].map(customer_signup)).all()
assert transactions['transaction_date'].ge(transactions['customer_id'].map(customer_signup)).all()
assert quality_report['duplicate_rows'].eq(0).all()
quality_report


,dataset,raw_rows,clean_rows,rows_removed,duplicate_rows,missing_cells
0,customers,5060,5000,60,0,189
1,campaigns,41,39,2,0,0
2,responses,15075,7477,7598,0,0
3,support_tickets,8050,6601,1449,0,5799
4,transactions,12060,9862,2198,0,0


## 9. Export Cleaned Datasets
Save the processed datasets and quality report to the `data/processed/` directory for analysis.

In [10]:
os.makedirs("../data/processed", exist_ok=True)

customers.to_csv("../data/processed/customers_clean.csv", index=False)
campaigns.to_csv("../data/processed/campaigns_clean.csv", index=False)
responses.to_csv("../data/processed/responses_clean.csv", index=False)
support_tickets.to_csv("../data/processed/support_tickets_clean.csv", index=False)
transactions.to_csv("../data/processed/transactions_clean.csv", index=False)
quality_report.to_csv("../data/processed/clean_quality_report.csv", index=False)
temporal_exclusion_report.to_csv("../data/processed/temporal_exclusion_report.csv", index=False)
print("Clean datasets and quality report exported successfully.")

Clean datasets and quality report exported successfully.
